# ReAct Agent

**Agentic AI** คือ ระบบ AI ที่ไม่ได้แค่ตอบคำถามแบบ Passive แต่สามารถตัดสินใจและลงมือทำ (Active) เพื่อให้บรรลุเป้าหมายที่ได้รับมอบหมาย 


**ReAct (Reason + Act)** คือ รูปแบบการคิดของ Agent แบบหนึ่ง ประกอบด้วย 2 ส่วนหลัก:
* Reason (Reasoning): การคิดวิเคราะห์ วางแผน และสรุปสถานการณ์ปัจจุบัน
* Act (Acting): การลงมือทำโดยใช้ Tools ที่มี หรือตอบกลับผู้ใช้


อ่านเพิ่มเติม https://huggingface.co/blog/VirtualOasis/agents-vs-workflows-en

# Task: The Vibe Coder

**เป้าหมาย**  

สร้าง Agent ที่รับคำสั่งภาษาไทยแบบบ้านๆ (Natural Language) แล้วไปเขียน Python Code พร้อมรันผลลัพธ์ให้ทันที!

**ตัวอย่าง**

* ผู้ใช้สั่งงาน: "ช่วยสร้างกราฟ Sine wave สีแดงให้หน่อย"
* Agent (Think): คิดว่าต้องใช้ Python และ Library matplotlib
* Agent (Act): เขียน Code และส่งไปที่ python_interpreter
* Agent (Observe): ดูผลลัพธ์ว่า Error หรือไม่
* Agent (Final Answer): ส่ง Code ที่สมบูรณ์และคำอธิบายกลับมา

In [28]:
from IPython.display import Image
from IPython.core.display import HTML 
Image(url= "./Examples/react-agent.png", width=300)

In [29]:
# graph TD
#     %% Node Definitions
#     Start((START))
#     AgentNode[<b>CallModel</b><br/>Think]
#     ActionNode[<b>CallTool</b><br/>Act: python_interpreter]
#     Decision{Should<br/>Continue?}
#     EndNode((END))

#     %% Workflow Connections
#     Start --> AgentNode
#     AgentNode --> Decision

#     %% ReAct Loop Logic
#     Decision -- "Act" --> ActionNode
#     ActionNode -- "Observe" --> AgentNode

#     %% Final Response
#     Decision -- "Final Answer" --> EndNode

#     %% Styling
#     style AgentNode fill:#e1f5fe,stroke:#01579b,stroke-width:2px
#     style ActionNode fill:#fff3e0,stroke:#e65100,stroke-width:2px
#     style Decision fill:#f3e5f5,stroke:#4a148c,stroke-width:2px

In [30]:
# !uv pip install langchain-ollama langgraph pydantic

In [31]:
from typing import TypedDict, Annotated, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama

In [32]:
# Adapt code from: https://ai.google.dev/gemini-api/docs/langgraph-example

In [33]:
llm = ChatOllama(model="scb10x/typhoon2.5-qwen3-4b")

In [34]:
import sys
import io
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

@tool
def python_interpreter(code: str):
    """
    Executes Python code and returns the standard output.
    Use this to solve math, process data, or verify logic.
    """
    stdout_buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = stdout_buffer
    
    try:
        # รันโค้ดที่ Agent ส่งมา
        exec(code, {})
        output = stdout_buffer.getvalue()
        print(f"Execution Successful.")
        return output
    except Exception as e:
        return f"Execution Failed. Error: {str(e)}"
    finally:
        # คืนค่า Standard Output กลับไปที่เดิม
        sys.stdout = old_stdout

tools = [python_interpreter]
tool_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)

In [35]:
print(python_interpreter.func("print(3, 1+2)"))

3 3



In [36]:
code = '''
for i in range(10):
    print(i)
'''
print(python_interpreter.func(code))

0
1
2
3
4
5
6
7
8
9



### Init State

In [42]:
from typing import Annotated,Sequence, TypedDict

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages  # helper function to add messages to the state


class AgentState(TypedDict):
    """The state of the agent."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    number_of_steps: int

In [43]:
from langchain_core.messages import ToolMessage
from langchain_core.runnables import RunnableConfig

tools_by_name = {tool.name: tool for tool in tools}

# Define our tool node
def CallTool(state: AgentState):
    outputs = []
    # Iterate over the tool calls in the last message
    for tool_call in state["messages"][-1].tool_calls:
        # Get the tool by name
        tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(
            ToolMessage(
                content=tool_result,
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    print("[CallTool]...")
    return {"messages": outputs}

def CallModel(
    state: AgentState,
    config: RunnableConfig,
):
    # Invoke the model with the system prompt and the messages
    response = llm_with_tools.invoke(state["messages"], config)
    # This returns a list, which combines with the existing messages state
    # using the add_messages reducer.
    print("[CallModel]...")
    return {"messages": [response]}


# Define the conditional edge that determines whether to continue or not
def should_continue(state: AgentState):
    messages = state["messages"]
    print("should_continue??", messages[-1])
    # If the last message is not a tool call, then finish
    if not messages[-1].tool_calls:
        return "end"
    # default to continue
    return "continue"

### Build the workflow

In [44]:
from langgraph.graph import StateGraph, END

# Define a new graph with our state
workflow = StateGraph(AgentState)

# 1. Add the nodes
workflow.add_node("llm", CallModel)
workflow.add_node("tools",  CallTool)


# 2. Set the entrypoint as `agent`, this is the first node called
workflow.set_entry_point("llm")

# 3. Add a conditional edge after the `llm` node is called.
workflow.add_conditional_edges(
    # Edge is used after the `llm` node is called.
    "llm",
    # The function that will determine which node is called next.
    should_continue,
    # Mapping for where to go next, keys are strings from the function return,
    # and the values are other nodes.
    # END is a special node marking that the graph is finish.
    {
        # If `tools`, then we call the tool node.
        "continue": "tools",
        # Otherwise we finish.
        "end": END,
    },
)
# 4. Add a normal edge after `tools` is called, `llm` node is called next.
workflow.add_edge("tools", "llm")

# Now we can compile and visualize our graph
graph = workflow.compile()

In [18]:
from datetime import datetime
system_prompt = SystemMessage(content="You are a ReAct agent. First 'Think' about the plan, then 'Act' using tools, and 'Observe' the result before giving the 'Final Answer'.")
inputs = {"messages": [
    system_prompt,
    ("user", f"Give me a python code to generate all prime numbers below 100")
]}

# call our graph with streaming to see the steps
for state in graph.stream(inputs, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()

================================ Human Message =================================

Give me a python code to generate all prime numbers below 100
[CallModel]...
should_continue?? content='' additional_kwargs={} response_metadata={'model': 'scb10x/typhoon2.5-qwen3-4b', 'created_at': '2026-03-08T16:06:36.611736Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13349541000, 'load_duration': 92735917, 'prompt_eval_count': 195, 'prompt_eval_duration': 1309419292, 'eval_count': 133, 'eval_duration': 11912181948, 'logprobs': None, 'model_name': 'scb10x/typhoon2.5-qwen3-4b', 'model_provider': 'ollama'} id='lc_run--019cce33-211c-7160-837b-27927d110ad1-0' tool_calls=[{'name': 'python_interpreter', 'args': {'code': 'def is_prime(n):\n    if n < 2:\n        return False\n    if n == 2:\n        return True\n    if n % 2 == 0:\n        return False\n    for i in range(3, int(n**0.5) + 1, 2):\n        if n % i == 0:\n            return False\n    return True\n\nprimes = [n for n in range(2, 10

In [19]:
# https://programming.in.th/tasks/0002

query = '''
Give me a python code to solve this task:

โจทย์จงหาค่าน้อยที่สุด และค่ามากที่สุด จากข้อมูลที่กำหนดให้ และแสดงผลออกทางจอภาพ

ข้อมูลนำเข้า
บรรทัดแรก จำนวนเต็มบวก n (1 ≤ n ≤ 1 000) บ่งบอกถึงจำนวนข้อมูลที่โจทย์กำหนดให้
บรรทัดที่ 2 ถึง n + 1 จำนวนเต็ม Ai
เป็นข้อมูลทั้งหมด (−2 000 000 000 ≤ Ai ≤ 2 000 000 000)

ข้อมูลส่งออก
บรรทัดแรก จำนวนเต็ม m แสดงจำนวนที่มีค่าน้อยที่สุดในชุดข้อมูลที่โจทย์กำหนด
บรรทัดที่สอง จำนวนเต็ม M แสดงจำนวนที่มีค่ามากที่สุดในชุดข้อมูลที่โจทย์กำหนด

ตัวอย่างข้อมูลนำเข้า 
5
1
2
3
4
5

ตัวอย่างข้อมูลส่งออก
1
5
'''


inputs = {"messages": [system_prompt, ("user", query)]}

for state in graph.stream(inputs, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()
    

================================ Human Message =================================


Give me a python code to solve this task:

โจทย์จงหาค่าน้อยที่สุด และค่ามากที่สุด จากข้อมูลที่กำหนดให้ และแสดงผลออกทางจอภาพ

ข้อมูลนำเข้า
บรรทัดแรก จำนวนเต็มบวก n (1 ≤ n ≤ 1 000) บ่งบอกถึงจำนวนข้อมูลที่โจทย์กำหนดให้
บรรทัดที่ 2 ถึง n + 1 จำนวนเต็ม Ai
เป็นข้อมูลทั้งหมด (−2 000 000 000 ≤ Ai ≤ 2 000 000 000)

ข้อมูลส่งออก
บรรทัดแรก จำนวนเต็ม m แสดงจำนวนที่มีค่าน้อยที่สุดในชุดข้อมูลที่โจทย์กำหนด
บรรทัดที่สอง จำนวนเต็ม M แสดงจำนวนที่มีค่ามากที่สุดในชุดข้อมูลที่โจทย์กำหนด

ตัวอย่างข้อมูลนำเข้า 
5
1
2
3
4
5

ตัวอย่างข้อมูลส่งออก
1
5

[CallModel]...
should_continue?? content='' additional_kwargs={} response_metadata={'model': 'scb10x/typhoon2.5-qwen3-4b', 'created_at': '2026-03-08T16:07:22.641562Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10132682875, 'load_duration': 92791834, 'prompt_eval_count': 426, 'prompt_eval_duration': 2926117750, 'eval_count': 72, 'eval_duration': 7089695957, 'logprobs': 

 5
 1
 2
 3
 4
 5


[CallTool]...
================================= Tool Message =================================
Name: python_interpreter

1
5

[CallModel]...
should_continue?? content='The minimum value is **1** and the maximum value is **5**.' additional_kwargs={} response_metadata={'model': 'scb10x/typhoon2.5-qwen3-4b', 'created_at': '2026-03-08T16:07:35.251735Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2160315375, 'load_duration': 91080125, 'prompt_eval_count': 513, 'prompt_eval_duration': 596197375, 'eval_count': 17, 'eval_duration': 1464996918, 'logprobs': None, 'model_name': 'scb10x/typhoon2.5-qwen3-4b', 'model_provider': 'ollama'} id='lc_run--019cce34-31e1-7372-8b03-a0d0b1bfd236-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 513, 'output_tokens': 17, 'total_tokens': 530}
================================== Ai Message ==================================

The minimum value is **1** and the maximum value is **5**.


In [40]:
from datetime import datetime
system_prompt = SystemMessage(content="You are a ReAct agent. First 'Think' about the plan, then 'Act' using tools, and 'Observe' the result before giving the 'Final Answer'.")
task = f'''
Find a 3x3 perfect square with the following format:

? ? ?
? 2 ?
? ? ?
'''
user_prompt = ("user", task)
inputs = {"messages": [
    system_prompt,
    user_prompt,
]}

# call our graph with streaming to see the steps
for state in graph.stream(inputs, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()

================================ Human Message =================================


Find a 3x3 perfect square with the following format:

? ? ?
? 2 ?
? ? ?

[CallModel]...
should_continue?? content='' additional_kwargs={} response_metadata={'model': 'scb10x/typhoon2.5-qwen3-4b', 'created_at': '2026-03-09T01:44:02.333031Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18278526667, 'load_duration': 1736168959, 'prompt_eval_count': 202, 'prompt_eval_duration': 1069454791, 'eval_count': 495, 'eval_duration': 15360842345, 'logprobs': None, 'model_name': 'scb10x/typhoon2.5-qwen3-4b', 'model_provider': 'ollama'} id='lc_run--019cd043-b4b1-7321-9fca-8d687933601d-0' tool_calls=[{'name': 'python_interpreter', 'args': {'code': "def find_perfect_square():\n    # We're looking for a 3x3 perfect square with the center digit being 2\n    # A perfect square means the number is the square of an integer\n    # The number has the form ? ? ?\n    #       ? 2 ?\n    #       ? ? ?\n    \n    # Let's

In [41]:
# https://programming.in.th/tasks/1013
query = '''
Give me a python code to solve this task:

Expression
ในการแทนนิพจน์(expression) ใด ๆด้วยฟังก์ชัน นิพจน์หลักจะถูกแบ่งเป็นนิพจน์ย่อยๆ ด้วยตัวดำเนินการ(operator)
ต่าง ๆดังนี้ การบวก “+”, วงเล็บ “( )”, การคูณ “ * ” และการยกกำลัง “^” โดยสามารถเขียนแทนด้วยฟังก์ชันได้
ดังนี้op(i, e) โดยที่ e หมายถึงนิพจน์ทางคณิตศาสตร์ใด ๆ ซึ่งสามารถถูกแบ่งเป็นนิพจน์ย่อย ๆ ได้โดยใช้ตัวดำเนิน
การที่มีลำดับความสำคัญในการทำงาน (priority) ต่ำสุดในนิพจน์นั้น และ i คือลำดับของนิพจน์ย่อยนั้นๆ ตัวอย่าง
เช่น นิพจน์a*b+b*c+c*d สามารถแบ่งเป็นสามนิพจน์ย่อย โดยมีนิพจน์ย่อยที่ 1 คือ a*b, นิพจน์ย่อยที่ 2 คือ b*c
และนิพจน์ย่อยที่ 3 คือ c*d เนื่องจากตัวดำเนินการ “+” มีความสำคัญต่ำสุดในการทำงานในนิพจน์นี้
กำหนดให้ลำดับความสำคัญในการทำงานของตัวดำเนินการจากมากสุดไปน้อยสุดมีดังนี้“( )”, “^”, “ * ” และ “+”
ตามลำดับ

วัตถุประสงค์ของฟังก์ชันแทนนิพจน์ คือ ต้องการแทนนิพจน์ย่อยด้วยฟังก์ชันเพื่อใช้ในการคำนวณ เช่น op(2, e) แทน
นิพจน์ย่อยลำดับที่สองของ e ที่กำหนดให้ข้างบน (a*b+b*c+c*d) ซึ่งจะได้op(2, e) = b*c

ตัวอย่าง
กำหนดให้นิพจน์ p มีค่าดังนี้: a^b*c+(d*c)^f*z+b ,สามารถแทนนิพจน์ย่อยใดๆ ของ p ด้วยฟังก์ชันได้ดังนี้
• op(3, p) = b
• op(1, op(3, p)) = b
• op(2, p) = (d*c)^f*z
• op(1, op(2, p)) = (d*c)^f
• op(1, op(1, op(2, p))) = (d*c)
• op(1, op(1, op(1, op(2, p)))) = d*c
• op(2, op(1, op(1, op(2, p)))) = null (ไม่มีคำตอบ)
• op(2, op(2, p)) = z

โจทย์จงเขียนโปรแกรมเพื่อรับข้อมูลนิพจน์ p ใด ๆ และฟังก์ชันคำถาม จากนั้นคำนวณหานิพจน์ย่อยของ p ที่
สอดคล้องกับฟังก์ชันที่กำหนด

ข้อมูลนำเข้า
บรรทัดแรก รับนิพจน์หลัก p ที่ประกอบด้วยตัวอักษรภาษาอังกฤษพิมพ์เล็ก a ถึง z และตัวดำเนินการเขียนติดกัน
โดยไม่มีช่องว่าง รับประกันว่าความยาวตัวอักษรและตัวดำเนินการรวมกันไม่เกิน 64 ตัว
บรรทัดที่สอง รับเลขจำนวนเต็มบวก n (1 ≤ n ≤ 10) แสดงจำนวนฟังก์ชันคำถาม n ฟังก์ชัน
บรรทัดที่ 3 ถึง n + 2 บรรทัดที่ i + 2 ให้รับฟังก์ชันคำถามที่ i โดยแต่ละบรรทัดประกอบด้วยเลขจำนวนเต็มบวกอยู่
ระหว่าง 1 ถึง 9 คั่นด้วยช่องว่าง 1 ช่อง และปิดท้ายด้วย 0
ตัวอย่างข้อมูลนำเข้าในบรรทัดที่ 3 ถึง n + 2
ข้อมูลนำเข้า 3 0 หมายถึงฟังก์ชัน op(3, p)
ข้อมูลนำเข้า 2 1 1 0 หมายถึงฟังก์ชัน op(1, op(1, op(2, p)))
ข้อมูลนำเข้า 1 2 2 0 หมายถึงฟังก์ชัน op(2, op(2, op(1, p)))

ข้อมูลส่งออก
มีn บรรทัด บรรทัดที่ i ให้แสดงฟังก์ชันและนิพจน์ย่อยที่สอดคล้องกับฟังก์ชันคำถามที่ i โดยในแต่ละบรรทัดของ
ข้อมูลส่งออกจะต้องไม่มีการเว้นวรรคใดๆ กรณีที่ไม่มีคำตอบให้แสดง “null”


ตัวอย่างข้อมูลนำเข้า
a*b^c+d*e^f
2
1 0
2 0

ตัวอย่างข้อมูลส่งออก
op(1,p)=a*b^c
op(2,p)=d*e^f
'''


inputs = {"messages": [system_prompt, ("user", query)]}

for state in graph.stream(inputs, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()
    

================================ Human Message =================================


Give me a python code to solve this task:

Expression
ในการแทนนิพจน์(expression) ใด ๆด้วยฟังก์ชัน นิพจน์หลักจะถูกแบ่งเป็นนิพจน์ย่อยๆ ด้วยตัวดำเนินการ(operator)
ต่าง ๆดังนี้ การบวก “+”, วงเล็บ “( )”, การคูณ “ * ” และการยกกำลัง “^” โดยสามารถเขียนแทนด้วยฟังก์ชันได้
ดังนี้op(i, e) โดยที่ e หมายถึงนิพจน์ทางคณิตศาสตร์ใด ๆ ซึ่งสามารถถูกแบ่งเป็นนิพจน์ย่อย ๆ ได้โดยใช้ตัวดำเนิน
การที่มีลำดับความสำคัญในการทำงาน (priority) ต่ำสุดในนิพจน์นั้น และ i คือลำดับของนิพจน์ย่อยนั้นๆ ตัวอย่าง
เช่น นิพจน์a*b+b*c+c*d สามารถแบ่งเป็นสามนิพจน์ย่อย โดยมีนิพจน์ย่อยที่ 1 คือ a*b, นิพจน์ย่อยที่ 2 คือ b*c
และนิพจน์ย่อยที่ 3 คือ c*d เนื่องจากตัวดำเนินการ “+” มีความสำคัญต่ำสุดในการทำงานในนิพจน์นี้
กำหนดให้ลำดับความสำคัญในการทำงานของตัวดำเนินการจากมากสุดไปน้อยสุดมีดังนี้“( )”, “^”, “ * ” และ “+”
ตามลำดับ

วัตถุประสงค์ของฟังก์ชันแทนนิพจน์ คือ ต้องการแทนนิพจน์ย่อยด้วยฟังก์ชันเพื่อใช้ในการคำนวณ เช่น op(2, e) แทน
นิพจน์ย่อยลำดับที่สองของ e ที

### Alternative Implementation

see: https://docs.langchain.com/oss/python/langchain/agents

In [23]:
from langchain.agents import create_agent
from langchain_core.tools import tool

tools = [python_interpreter]

# 2. Create the agent
# You can pass a model name string or a ChatOpenAI instance
agent = create_agent(
    model=llm, 
    tools=tools,
    system_prompt="You are a helpful assistant that uses tools to answer questions."
)

# 3. Run the agent
# The new agent returns a state containing a list of messages
result = agent.invoke({"messages": [{"role": "user", "content": query}]})


 5
 1
 24
 3
 2
 3


The smallest value is 1 and the largest value is 24.


In [27]:
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================


Give me a python code to solve this task:

โจทย์จงหาค่าน้อยที่สุด และค่ามากที่สุด จากข้อมูลที่กำหนดให้ และแสดงผลออกทางจอภาพ

ข้อมูลนำเข้า
บรรทัดแรก จำนวนเต็มบวก n (1 ≤ n ≤ 1 000) บ่งบอกถึงจำนวนข้อมูลที่โจทย์กำหนดให้
บรรทัดที่ 2 ถึง n + 1 จำนวนเต็ม Ai
เป็นข้อมูลทั้งหมด (−2 000 000 000 ≤ Ai ≤ 2 000 000 000)

ข้อมูลส่งออก
บรรทัดแรก จำนวนเต็ม m แสดงจำนวนที่มีค่าน้อยที่สุดในชุดข้อมูลที่โจทย์กำหนด
บรรทัดที่สอง จำนวนเต็ม M แสดงจำนวนที่มีค่ามากที่สุดในชุดข้อมูลที่โจทย์กำหนด

ตัวอย่างข้อมูลนำเข้า 
5
1
2
3
4
5

ตัวอย่างข้อมูลส่งออก
1
5

================================== Ai Message ==================================
Tool Calls:
  python_interpreter (c26b1a8e-056b-478e-b0fc-500036b65cca)
 Call ID: c26b1a8e-056b-478e-b0fc-500036b65cca
  Args:
    code: n = int(input())
numbers = []
for _ in range(n):
    numbers.append(int(input()))

min_val = min(numbers)
max_val = max(numbers)

print(min_val)
print(max_val)
======

# Plan-and-Execute Agent

**Plan-and-Execute** คือ รูปแบบการคิดของ Agent อีกแบบหนึ่ง จาก [Wang et al. (2023)](https://arxiv.org/abs/2305.04091)

see code example: https://github.com/langchain-ai/langgraph/blob/23961cff61a42b52525f3b20b4094d8d2fba1744/docs/docs/tutorials/plan-and-execute/plan-and-execute.ipynb

# Recursive Language Models

Recursive Language Models (RLMs), an inference strategy where language models can decompose and recursively interact with input context of unbounded length through REPL environments.
ref: https://alexzhang13.github.io/blog/2025/rlm/